In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# --- 1. Set Up Tiny Dimensions ---
batch_size = 1
seq_len = 3              # We have 3 tokens in our sequence
emb_dim = 4              
num_experts = 3          # We have 3 experts in total
num_experts_per_tok = 2  # The router will pick the top 2 experts for each token

print("--- 1. Input ---")
x = torch.randn(batch_size, seq_len, emb_dim)
print(f"Input 'x' shape: {x.shape} (batch, seq_len, emb_dim)")

# --- 2. Routing (The Gate) ---
# We simulate the gate's linear layer with matrix multiplication
gate_weights = torch.randn(emb_dim, num_experts)
scores = x @ gate_weights # Shape: (1, 3, 3) (batch_size, tokens,  num_experts)

# Find top k experts and calculate probabilities
topk_scores, topk_indicies = torch.topk(scores, num_experts_per_tok, dim=-1)
topk_probs = torch.softmax(topk_scores, dim=-1)

print("\n--- 2. Routing Results ---")
print(f"Top-2 Indices (Which experts were chosen for each token):\n{topk_indicies.squeeze(0)}")
print(f"Top-2 Probabilities:\n{topk_probs.squeeze(0)}")

# --- 3. Flattening for Processing ---
# Flatten the batch and sequence length together so we just have a list of tokens
x_flat = x.reshape(-1, emb_dim) 
topk_indicies_flat = topk_indicies.reshape(-1, num_experts_per_tok)
topk_probs_flat = topk_probs.reshape(-1, num_experts_per_tok)

# Create a blank canvas of zeros to store the final results
out_flat = torch.zeros_like(x_flat)

unique_experts = torch.unique(topk_indicies_flat)
print(f"\n--- 3. Experts to Process: {unique_experts.tolist()} ---")

# --- 4. The Expert Loop ---
for expert_id in unique_experts:
    expert_id = int(expert_id.item())
    print(f"\n=== Processing Expert {expert_id} ===")

    # Find which tokens chose this expert
    # mask
    # Shape: (3, 2)
    # [
        # [ True, False], # For Token 0, Expert 0 is in slot 0
        # [False,  True], # For Token 1, Expert 0 is in slot 1
        # [False, False], # For Token 2, Expert 0 was not chosen
    # ]

    # token_mask
    # Shape: (3,)
    # [
    #    True,  # Token 0 had a True? Yes.
    #    True,  # Token 1 had a True? Yes.
    #   False,  # Token 2 had a True? No.
    # ]
    mask = topk_indicies_flat == expert_id #  where the expert slot is (if there is at all) for this specific token
    token_mask = mask.any(dim=-1) # which tokens want this expert. Is this corect?
    # selected_idx tells us the exact row numbers of the tokens that need this expert
    selected_idx = token_mask.nonzero(as_tuple=False).squeeze(-1)
    print(f"Tokens selecting Expert {expert_id} (Row Indices): {selected_idx.tolist()}")

    if selected_idx.numel() == 0:
        continue

    # Extract ONLY the tokens that selected this expert
    expert_input = x_flat.index_select(0, selected_idx)

    # SIMULATING THE EXPERT (SwiGLU -> Linear)
    # Instead of running real linear layers, we just generate dummy output 
    # of the same shape to represent the data coming out of the expert.
    expert_out = torch.randn_like(expert_input)

    # --- Deciphering the Probabilities (The "Slot Indices" part) ---
    mask_selected = mask[selected_idx]
    
    # Where in the top-k list was this expert? Slot 0 (1st choice) or Slot 1 (2nd choice)?
    slot_indicies = mask_selected.int().argmax(dim=-1, keepdim=True)
    
    # Go into the probabilities tensor and grab the exact probability for that slot
    selected_probs = torch.gather(
        topk_probs_flat.index_select(0, selected_idx), dim=-1, index=slot_indicies
    ).squeeze(-1)

    print(f"Slot indices for these tokens (0=1st choice, 1=2nd choice): {slot_indicies.squeeze(-1).tolist()}")
    print(f"Routing probabilities fetched for weighting: {selected_probs.tolist()}")

    # Weight the expert's output by the routing probability
    weighted_out = expert_out * selected_probs.unsqueeze(-1)
    
    # Add the weighted output back into our blank canvas at the correct row indices
    out_flat.index_add_(0, selected_idx, weighted_out)

# --- 5. Reshape to Original ---
final_output = out_flat.reshape(batch_size, seq_len, emb_dim)
print("\n--- 5. Final Output ---")
print(f"Final shape: {final_output.shape}")

--- 1. Input ---
Input 'x' shape: torch.Size([1, 3, 4]) (batch, seq_len, emb_dim)

--- 2. Routing Results ---
Top-2 Indices (Which experts were chosen for each token):
tensor([[0, 2],
        [2, 1],
        [2, 0]])
Top-2 Probabilities:
tensor([[0.5506, 0.4494],
        [0.9621, 0.0379],
        [0.5534, 0.4466]])

--- 3. Experts to Process: [0, 1, 2] ---

=== Processing Expert 0 ===
tensor([[0, 2],
        [2, 1],
        [2, 0]])
Tokens selecting Expert 0 (Row Indices): [0, 2]
Slot indices for these tokens (0=1st choice, 1=2nd choice): [0, 1]
Routing probabilities fetched for weighting: [0.5506319999694824, 0.44662991166114807]

=== Processing Expert 1 ===
tensor([[0, 2],
        [2, 1],
        [2, 0]])
Tokens selecting Expert 1 (Row Indices): [1]
Slot indices for these tokens (0=1st choice, 1=2nd choice): [1]
Routing probabilities fetched for weighting: [0.03786708042025566]

=== Processing Expert 2 ===
tensor([[0, 2],
        [2, 1],
        [2, 0]])
Tokens selecting Expert 2 (Ro